# Where — and when — does a diffusion robot policy look?
**Faithful visual attribution for RDT-1B on ManiSkill3 (pilot study)**

This notebook runs a complete, small version of the study planned in
[`docs/GAMEPLAN.md`](https://github.com/rdgbrandon/rdt-igtesting/blob/cvpr-xai/docs/GAMEPLAN.md) and produces the
paper's tables and figures on pilot data.

| Stage | What it measures | Pilot size | Est. A100 time |
|---|---|---|---|
| **A** Rollouts | RDT success rate with the official protocol (should match 77% / 74% / 100%) | 3 tasks × 16 episodes | ~10 min |
| **B** Benchmark | 11 attribution methods at matched compute: deletion / insertion faithfulness, seed stability | 60 moments | ~60–90 min |
| **C** Denoising time | *When* in the 5 denoising steps vision is used (exact per-step Expected-IG split + knockout) | 30 moments | ~10 min |
| **D** Closed loop | Success when the top-10% attributed patches are hidden **while the robot acts** | 2 tasks × 6 conditions × 10 eps | ~60–90 min |
| **E** Sanity | Maps must change when RDT's weights are randomized (Adebayo et al. 2018) | 8 moments | ~10 min |

Times are estimates; cell 4 measures the real speed and prints a projection before anything long starts.

**How to run:** `Runtime → Change runtime type → A100 GPU`, then `Runtime → Run all`.
Everything is saved to Google Drive as it goes and every stage resumes where it stopped, so after a
disconnect just *Run all* again.

**Methods compared** (all explain the same scalar: the executed action chunk projected onto
*plan − plan-without-vision*):
random · DAAM (cross-attention, Tang '23) · GAE (gradient-weighted attention, Chefer '21) · gradient ·
grad×input · occlusion · ISS (random masking, ICML '26) · IG zero baseline · IG background baseline ·
BlurIG (this repo's original method) · **Expected IG over sampler noise (ours)**.

In [ ]:
#@title 1. Setup: GPU, Google Drive, install (~3 min). Re-running this cell always pulls the latest fixes.
import os, sys, subprocess, importlib
OUT_DRIVE = '/content/drive/MyDrive/rdt-xai/pilot1'  #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/rdt-igtesting'):
    subprocess.run('git clone -q -b cvpr-xai https://github.com/rdgbrandon/rdt-igtesting /content/rdt-igtesting',
                   shell=True, check=True)
subprocess.run('cd /content/rdt-igtesting && git fetch -q origin cvpr-xai && git checkout -q cvpr-xai && git pull -q origin cvpr-xai',
               shell=True, check=True)
sys.path.insert(0, '/content/rdt-igtesting')
import rdt_xai.colab as colab
importlib.reload(colab)
OUT = colab.setup(OUT_DRIVE)
sync = colab.sync
import time, json

In [ ]:
#@title 2. Download models (~6 GB, ~2 min): RDT-1B ManiSkill fine-tune, SigLIP-SO400M, language embeddings
import shutil
from huggingface_hub import hf_hub_download
D = '/content/rdt-data'
for sub in ('rdt', 'siglip', 'lang'):
    os.makedirs(f'{D}/{sub}', exist_ok=True)
for f in ['config.json', 'preprocessor_config.json', 'special_tokens_map.json', 'spiece.model',
          'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']:
    hf_hub_download('google/siglip-so400m-patch14-384', f, local_dir=f'{D}/siglip')
hf_hub_download('robotics-diffusion-transformer/maniskill-model', 'rdt/mp_rank_00_model_states.pt', local_dir=D)
for t in ['PickCube', 'StackCube', 'PushCube', 'PegInsertionSide', 'PlugCharger']:
    p = hf_hub_download('robotics-diffusion-transformer/maniskill-model', f'lang_embeds/text_embed_{t}-v1.pt', local_dir=D)
    if not os.path.exists(f'{D}/lang/text_embed_{t}-v1.pt'):
        shutil.copy(p, f'{D}/lang/text_embed_{t}-v1.pt')
os.system(f'du -sh {D}/*')

In [ ]:
#@title 3. Load RDT-1B and verify: exact match to RDT's own sampler, rendering works
import numpy as np, torch
from PIL import Image
from IPython.display import display, Markdown
from rdt_xai.policy import RDTPolicy, TASK2LANG
from rdt_xai.rollout import make_env, _render

pol = RDTPolicy()
print(f'device={pol.device} dtype={pol.dtype}  RDT params={sum(p.numel() for p in pol.rdt.parameters())/1e9:.2f}B')

# (a) the cached, slot-aware sampler reproduces RDT's predict_action on the official 6-slot input
rng = np.random.default_rng(0)
ip, ic = (Image.fromarray(rng.integers(0, 255, (512, 512, 3), dtype=np.uint8)) for _ in range(2))
q = np.array([0.0, 0.4, 0.0, -2.0, 0.0, 2.4, 0.8, 0.04], dtype=np.float32)
with torch.no_grad():
    lang = pol.lang('PickCube-v1'); noise = pol.noise(123)
    e6 = pol.encode([ip, None, None, ic, None, None]).reshape(1, -1, pol.img_dim)
    _randn = torch.randn; torch.randn = lambda *a, **k: noise.clone()
    ref = pol.rdt.predict_action(lang_tokens=lang, lang_attn_mask=torch.ones(lang.shape[:2], dtype=torch.bool, device=pol.device),
                                 img_tokens=e6, state_tokens=pol.state_token(q), action_mask=pol.elem_mask, ctrl_freqs=pol.ctrl_freqs)
    torch.randn = _randn
    e = pol.encode([ip, ic]); cond = pol.conditions('PickCube-v1', e[0:1], e[1:2], q)
    ours = pol.sample(cond, noise=noise)
err = (pol.unnormalize(ref) - pol.unnormalize(ours)).abs().max().item()
print(f'max |cached - reference| = {err:.2e} rad  ->', 'OK' if err < 5e-3 else 'MISMATCH (stop and report)')

# (b) rendering
env = make_env('PickCube-v1'); env.reset(seed=20241201); frame = _render(env); env.close()
print('render:', frame.size); display(frame.resize((256, 256)))

In [ ]:
#@title 4. Experiment size + measured speed -> projected runtime
TASKS = ['PickCube-v1', 'StackCube-v1', 'PushCube-v1']   #@param
N_EPISODES = 16           #@param {type:"integer"}
B_MOMENTS = 60            #@param {type:"integer"}
B_STABILITY_MOMENTS = 24  #@param {type:"integer"}
N_EVALS = 16              #@param {type:"integer"}
FAST = False              #@param {type:"boolean"}
C_MOMENTS = 30            #@param {type:"integer"}
D_TASKS = ['PickCube-v1', 'StackCube-v1']  #@param
D_EPISODES = 10           #@param {type:"integer"}
D_FRAC = 0.10             #@param {type:"number"}
E_MOMENTS = 8             #@param {type:"integer"}
METHODS = ['random', 'daam', 'gae', 'grad', 'gxi', 'occlusion', 'iss', 'ig_zero', 'ig_bg', 'blurig', 'eig']
D_CONDITIONS = [('none', 'none'), ('eig', 'top'), ('eig', 'bottom'), ('random', 'random'), ('daam', 'top'), ('blurig', 'top')]

from rdt_xai.attribution import Explainer
from rdt_xai.metrics import Faithfulness
ex = Explainer(pol, batch=8)
sync_cuda = lambda: torch.cuda.synchronize() if torch.cuda.is_available() else None
t = {}
sync_cuda(); t0 = time.time(); pol.act('PickCube-v1', ip, ic, q); sync_cuda(); t['policy call'] = time.time() - t0
for m in ('eig', 'occlusion', 'gae'):
    sync_cuda(); t0 = time.time(); ex.explain(cond, m, n_evals=N_EVALS, seed=1); sync_cuda(); t[m] = time.time() - t0
fm = Faithfulness.fast(pol) if FAST else Faithfulness(pol)
maps = {m: ex.explain(cond, m, n_evals=4, seed=1)['map'] for m in METHODS}
sync_cuda(); t0 = time.time(); fm.evaluate(cond, ip, ic, maps); sync_cuda(); t['faithfulness (all methods)'] = time.time() - t0
for k, v in t.items():
    print(f'{k:28s} {v:6.2f}s')
per_moment = 4 * t['eig'] + t['occlusion'] + 3 * t['gae'] + t['faithfulness (all methods)']
calls_per_ep = 12
proj = {'A rollouts': len(TASKS) * N_EPISODES * (calls_per_ep * t['policy call'] + 8),
        'B benchmark': B_MOMENTS * per_moment + B_STABILITY_MOMENTS * 4 * t['eig'],
        'C denoising': C_MOMENTS * (2.2 * t['eig'] + 12 * t['policy call']),
        'D closed loop': len(D_TASKS) * len(D_CONDITIONS) * D_EPISODES * (calls_per_ep * (t['policy call'] + t['eig'] / 2) + 8),
        'E sanity': E_MOMENTS * 4 * (2 * t['eig'] + t['gae'])}
for k, v in proj.items():
    print(f'projected {k:14s} {v/60:6.1f} min')
print(f'projected total          {sum(proj.values())/3600:6.1f} h')

In [ ]:
#@title A. Rollouts (official RDT ManiSkill protocol; resumable)
from rdt_xai import experiments as X
import pandas as pd
rows = X.rollouts(pol, TASKS, range(N_EPISODES), f'{OUT}/A')
df_a = pd.DataFrame(rows)
paper = {'PickCube-v1': .772, 'StackCube-v1': .74, 'PushCube-v1': 1.0, 'PegInsertionSide-v1': .132, 'PlugCharger-v1': .012}
s = df_a.groupby('task').success.agg(['mean', 'size']).rename(columns={'mean': 'ours', 'size': 'episodes'})
s['RDT paper'] = [paper.get(i) for i in s.index]
display(s)
print('synced', sync(OUT, OUT_DRIVE), 'files to Drive')

In [ ]:
#@title B. Attribution benchmark at matched compute (resumable)
moments = X.select_moments(f'{OUT}/A', tasks=TASKS, every=3, max_per_episode=3)
by_task = {}
for d, c in moments:
    by_task.setdefault(os.path.basename(d).split('_ep')[0], []).append((d, c))
balanced = [m for grp in zip(*[v for v in by_task.values()]) for m in grp] if len(by_task) > 1 else moments
balanced += [m for m in moments if m not in balanced]
B = balanced[:B_MOMENTS]
print(f'{len(moments)} candidate moments from successful episodes -> using {len(B)}')
X.benchmark(pol, B[:B_STABILITY_MOMENTS], METHODS, f'{OUT}/B', n_evals=N_EVALS, fast=FAST, stability=True)
print('synced', sync(OUT, OUT_DRIVE))
X.benchmark(pol, B[B_STABILITY_MOMENTS:], METHODS, f'{OUT}/B', n_evals=N_EVALS, fast=FAST, stability=False)
print('synced', sync(OUT, OUT_DRIVE))

In [ ]:
#@title C. When during denoising does vision matter? (exact per-step Expected-IG split + knockout)
X.denoise(pol, B[:C_MOMENTS], f'{OUT}/C', n_evals=N_EVALS)
print('synced', sync(OUT, OUT_DRIVE))

In [ ]:
#@title D. Closed-loop causal test: hide 10% of patches online, measure success (resumable)
X.closed_loop(pol, D_TASKS, range(D_EPISODES), D_CONDITIONS, f'{OUT}/D', frac=D_FRAC, n_evals=8)
print('synced', sync(OUT, OUT_DRIVE))

In [ ]:
#@title E. Model-randomization sanity check
X.sanity(pol, B[:E_MOMENTS], ['eig', 'ig_bg', 'blurig', 'grad', 'gae', 'daam'], f'{OUT}/E', n_evals=8)
print('synced', sync(OUT, OUT_DRIVE))

In [ ]:
#@title Results: tables + figures (also saved to Drive: report/RESULTS.md, report/*.png)
from rdt_xai.report import build_report
from IPython.display import Image as IPImage
txt = build_report(OUT)
display(Markdown(txt))
for f in ['fig_faithfulness_curves', 'fig_qualitative', 'fig_denoising_steps', 'fig_denoising_step_maps',
          'fig_closed_loop', 'fig_sanity']:
    p = f'{OUT}/report/{f}.png'
    if os.path.exists(p):
        print(f); display(IPImage(p, width=900))
print('synced', sync(OUT, OUT_DRIVE), 'files ->', OUT_DRIVE)

In [ ]:
#@title (Optional) Video: one successful episode with Expected-IG overlay at every decision
import imageio, matplotlib
from rdt_xai.rollout import load_episode
EPISODE = ''  #@param {type:"string"}
ep_dir = f'{OUT}/A/{EPISODE}' if EPISODE else sorted(d for d, _ in moments)[0]
meta = load_episode(ep_dir)
cmap = matplotlib.colormaps['inferno']
frames = []
for k, dp in enumerate(meta['decisions']):
    e = pol.encode([dp['prev'], dp['curr']])
    c = pol.conditions(meta['task'], e[0:1], e[1:2], dp['qpos8'])
    a = np.clip(ex.explain(c, 'eig', n_evals=8, seed=k)['map'], 0, None)
    a = a / (a.max() + 1e-12)
    up = np.pad(np.kron(a, np.ones((14, 14))), ((0, 6), (0, 6)))
    img = np.asarray(dp['curr'].resize((384, 384))) / 255.0
    al = (np.sqrt(up) * 0.8)[..., None]
    frames.append((255 * (img * (1 - al) + cmap(up)[..., :3] * al)).astype(np.uint8))
path = f'{OUT}/report/video_{os.path.basename(ep_dir)}_eig.mp4'
imageio.mimsave(path, frames, fps=2, macro_block_size=1)
print(path); sync(OUT, OUT_DRIVE)
from IPython.display import Video
display(Video(path, embed=True, width=384))

## Reading the results

* **A** — if success rates are far below the RDT paper (77% / 74% / 100%), stop: something about the
  simulator version or camera differs and nothing downstream is trustworthy.
* **B** — the headline table. Higher *deletion* and *insertion* = more faithful. `seed_spearman` near 1 =
  the map does not depend on the sampler's random noise. Paired Wilcoxon tests compare Expected IG with each method.
* **C** — attribution share and knockout effect per denoising step (step 1 = noisiest).
* **D** — closed loop: a faithful method's *top* patches should hurt success much more than *random*, and its
  *bottom* patches should hurt least.
* **E** — Spearman to the trained model's map after randomizing RDT blocks: it must drop, or the method is not
  explaining the model.

Send `report/RESULTS.md` (in your Drive folder) back for the next round of the plan.